# Task 2 — Quick Experiment Notebook

**Purpose:** Fast experiments to test new ideas quickly.

**Rules:**
- Use small dataset subset (e.g., 1000 images)
- Use few epochs (3-8)
- No cross-validation
- Quick feedback: "Does this idea help or not?"

**If idea works → move to Train_notebook.ipynb**  
**If idea fails → discard quickly**

**Runtime:** < 5-10 minutes per experiment

**Current Experiment:** Testing EfficientNet-B3 with 300x300 resolution


In [13]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch import amp
import importlib

import models
importlib.reload(models)
from models import create_efficientnet_b3
import datasets
importlib.reload(datasets)
from datasets import Task2TrainingDataset300, Task2TestDataset300

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Device: cuda


## Experiment Configuration

**Modify these for your experiment:**


In [14]:
# ============================================================
# EXPERIMENT CONFIG - Modify for your test
# ============================================================

# Quick training config
num_epochs = 10  # Keep it small!
batch_size = 64  # Reduced for B3 (larger model + higher resolution)
n_samples_to_use = 1000  # Use subset of data for speed

# What are you testing?
experiment_name = "test_efficientnet_b3_300x300"  # Name your experiment
test_label_smoothing = 0.2  # Example: testing different label smoothing

# Model config
learning_rate = 1e-3
weight_decay = 1e-4
use_amp = True

print(f"Experiment: {experiment_name}")
print(f"Using {n_samples_to_use} samples, {num_epochs} epochs")
print(f"Model: EfficientNet-B3, Resolution: 300x300")


Experiment: test_efficientnet_b3_300x300
Using 1000 samples, 10 epochs
Model: EfficientNet-B3, Resolution: 300x300


## Load Subset of Data


In [15]:
train_dataset = Task2TrainingDataset300()
val_dataset = Task2TestDataset300()

# Use subset for speed
all_indices = np.arange(len(train_dataset))
np.random.seed(42)
selected_indices = np.random.choice(all_indices, size=min(n_samples_to_use, len(all_indices)), replace=False)

# Split into train/val
split_idx = int(len(selected_indices) * 0.8)
train_idx = selected_indices[:split_idx]
val_idx = selected_indices[split_idx:]

train_subset = Subset(train_dataset, train_idx)
val_subset = Subset(val_dataset, val_idx)

train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train samples: {len(train_subset)}")
print(f"Val samples: {len(val_subset)}")


Train samples: 800
Val samples: 200


## Quick Training Loop


In [16]:
import time
from tqdm import tqdm

# Create model
print("[LOG] Creating model...")
model = create_efficientnet_b3(num_classes=10).to(device)
print(f"[LOG] Model created and moved to {device}")

# Optimizer and scheduler
print("[LOG] Setting up optimizer and scheduler...")
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
print(f"[LOG] Optimizer: Adam, LR: {learning_rate}, Weight Decay: {weight_decay}")

# Loss with your experiment parameter
criterion = nn.CrossEntropyLoss(label_smoothing=test_label_smoothing)
scaler = amp.GradScaler("cuda") if use_amp else None
print(f"[LOG] Loss: CrossEntropyLoss with label_smoothing={test_label_smoothing}")
print(f"[LOG] Mixed Precision (AMP): {use_amp}")

# Training
print(f"\n{'='*60}")
print(f"Starting experiment: {experiment_name}")
print(f"{'='*60}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print(f"Batch size: {batch_size}\n")

for epoch in range(num_epochs):
    epoch_start_time = time.time()
    print(f"\n[EPOCH {epoch+1}/{num_epochs}]")
    print(f"{'-'*60}")
    
    # Train
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    print(f"[TRAIN] Starting training phase...")
    train_start = time.time()
    
    for batch_idx, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False)):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        if use_amp:
            with amp.autocast("cuda"):
                pred = model(x)
                loss = criterion(pred, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
        
        train_loss += loss.item()
        _, predicted = pred.max(1)
        train_total += y.size(0)
        train_correct += predicted.eq(y).sum().item()
    
    scheduler.step()
    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    train_time = time.time() - train_start
    current_lr = scheduler.get_last_lr()[0]
    
    print(f"[TRAIN] Completed in {train_time:.2f}s")
    print(f"[TRAIN] Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, LR: {current_lr:.6f}")
    
    # Validate
    print(f"[VAL] Starting validation phase...")
    model.eval()
    correct = 0
    total = 0
    val_start = time.time()
    
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            pred = model(x)
            _, cls = pred.max(1)
            total += y.size(0)
            correct += (cls == y).sum().item()
    
    val_acc = correct / total
    val_time = time.time() - val_start
    epoch_time = time.time() - epoch_start_time
    
    print(f"[VAL] Completed in {val_time:.2f}s")
    print(f"[VAL] Accuracy: {val_acc:.4f} ({correct}/{total})")
    print(f"[EPOCH {epoch+1}] Total time: {epoch_time:.2f}s | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print(f"\n{'='*60}")
print(f"Experiment '{experiment_name}' completed!")
print(f"Final validation accuracy: {val_acc:.4f}")
print(f"{'='*60}")


[LOG] Creating model...
[LOG] Model created and moved to cuda
[LOG] Setting up optimizer and scheduler...
[LOG] Optimizer: Adam, LR: 0.001, Weight Decay: 0.0001
[LOG] Loss: CrossEntropyLoss with label_smoothing=0.2
[LOG] Mixed Precision (AMP): True

Starting experiment: test_efficientnet_b3_300x300
Train batches: 13, Val batches: 4
Batch size: 64


[EPOCH 1/10]
------------------------------------------------------------
[TRAIN] Starting training phase...


[TRAIN] Completed in 7.07s
[TRAIN] Loss: 1.8738, Acc: 0.4688, LR: 0.000976
[VAL] Starting validation phase...


[VAL] Completed in 5.14s
[VAL] Accuracy: 0.7650 (153/200)
[EPOCH 1] Total time: 12.21s | Train Loss: 1.8738 | Train Acc: 0.4688 | Val Acc: 0.7650

[EPOCH 2/10]
------------------------------------------------------------
[TRAIN] Starting training phase...


[TRAIN] Completed in 6.70s
[TRAIN] Loss: 1.2826, Acc: 0.8113, LR: 0.000905
[VAL] Starting validation phase...


[VAL] Completed in 4.93s
[VAL] Accuracy: 0.8450 (169/200)
[EPOCH 2] Total time: 11.63s | Train Loss: 1.2826 | Train Acc: 0.8113 | Val Acc: 0.8450

[EPOCH 3/10]
------------------------------------------------------------
[TRAIN] Starting training phase...


...

[VAL] Completed in 4.93s
[VAL] Accuracy: 0.9000 (180/200)
[EPOCH 8] Total time: 11.64s | Train Loss: 0.9205 | Train Acc: 0.9825 | Val Acc: 0.9000

[EPOCH 9/10]
------------------------------------------------------------
[TRAIN] Starting training phase...


[TRAIN] Completed in 6.59s
[TRAIN] Loss: 0.9049, Acc: 0.9962, LR: 0.000024
[VAL] Starting validation phase...


[VAL] Completed in 4.87s
[VAL] Accuracy: 0.8650 (173/200)
[EPOCH 9] Total time: 11.46s | Train Loss: 0.9049 | Train Acc: 0.9962 | Val Acc: 0.8650

[EPOCH 10/10]
------------------------------------------------------------
[TRAIN] Starting training phase...


[TRAIN] Completed in 6.67s
[TRAIN] Loss: 0.9043, Acc: 0.9950, LR: 0.000000
[VAL] Starting validation phase...


[VAL] Completed in 5.04s
[VAL] Accuracy: 0.8950 (179/200)
[EPOCH 10] Total time: 11.72s | Train Loss: 0.9043 | Train Acc: 0.9950 | Val Acc: 0.8950

Experiment 'test_efficientnet_b3_300x300' completed!
Final validation accuracy: 0.8950
